In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc pandas qiskit-ibm-catalog
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Quantum Portfolio Optimizer: Qiskit Function від Global Data Quantum


> **Note:** Qiskit Functions — це експериментальна функція, доступна лише користувачам тарифних планів IBM Quantum&reg; Premium Plan, Flex Plan та On-Prem (через IBM Quantum Platform API). Вони перебувають у стані попереднього релізу та можуть змінюватися.
# Огляд
Quantum Portfolio Optimizer — це Qiskit Function, яка вирішує задачу динамічної оптимізації портфеля, стандартну задачу у фінансах, що спрямована на перебалансування періодичних інвестицій між набором активів з метою максимізації прибутку та мінімізації ризиків. Використовуючи передові методи квантової оптимізації, ця функція спрощує процес так, щоб користувачі без будь-яких знань у сфері квантових обчислень могли скористатися її перевагами у пошуку оптимальних інвестиційних траєкторій. Ідеально підходить для портфельних менеджерів, дослідників у кількісних фінансах та індивідуальних інвесторів — цей інструмент дає змогу проводити бектестування торгових стратегій при оптимізації портфеля.
# Опис функції
Функція Quantum Portfolio Optimizer використовує алгоритм Варіаційного Квантового Власного Розкладача (VQE) для розв'язання задачі Квадратичної Незв'язаної Бінарної Оптимізації (QUBO), вирішуючи задачі динамічної оптимізації портфеля. Користувачам потрібно лише надати дані про ціни активів і задати інвестиційні обмеження, після чого функція запускає процес квантової оптимізації, який повертає набір оптимізованих інвестиційних траєкторій.

Процес складається з чотирьох основних етапів. Спочатку вхідні дані відображаються у квантово-сумісну задачу: будується QUBO для задачі динамічної оптимізації портфеля та перетворюється на квантовий оператор (Ізінговий гамільтоніан). Далі вхідна задача і алгоритм VQE адаптуються для роботи на квантовому залізі. Після цього алгоритм VQE запускається на квантовому залізі, і нарешті результати постобробляються для отримання оптимальних інвестиційних траєкторій. Система також включає шумозахисну постобробку на основі ([SQD](/guides/qiskit-addons-sqd)) для максимізації якості вихідних даних.

Ця Qiskit Function базується на [опублікованій статті](https://arxiv.org/abs/2412.19150) компанії Global Data Quantum.
![Візуалізація робочого процесу функції](../docs/images/guides/global-data-quantum-optimizer/function_workflow.svg)
# Вхідні дані
Вхідні аргументи функції описані в таблиці нижче. Дані про активи та інші параметри задачі є обов'язковими; крім того, можна задати налаштування VQE для налаштування процесу оптимізації.
| **Назва**              | **Тип**            | **Опис**                                                 | **Обов'язковий** | **За замовчуванням**                  | **Приклад**                              |
|------------------------|--------------------|----------------------------------------------------------|------------------|--------------------------------------|------------------------------------------|
| assets                 | json               | Словник із цінами активів                                | Так              | -                                    | -                                        |
| qubo_settings          | json               | Налаштування QUBO                                        | Так              | -                                    | Дивись приклади в таблиці нижче         |
| ansatz_settings        | json               | Налаштування ансатцу                                     | Ні               | `None`                                | Дивись приклади в таблиці нижче.         |
| optimizer_settings     | json               | Налаштування оптимізатора                                | Ні               | `None`                                | Дивись приклади в таблиці нижче.         |
| backend                | str                | Назва бекенду QPU                                        | Ні               |  -    | `"ibm_torino"`                           |
| previous_session_id    | list of str        | Список ідентифікаторів сесій для отримання даних із попередніх запусків[(*)](#1-note) | Ні               | Порожній список                      | `["session_id_1", "session_id_2"]`      |
| apply_postprocess      | bool               | Застосувати шумозахисну постобробку SQD                  | Ні               | `True`                                | `True`                                   |
| tags                   | list of strings    | Список тегів для ідентифікації експерименту              | Ні               | Порожній список                      | `["optimization", "quantum_computing"]`  |
<span id="1-note"></span>
*Щоб відновити виконання або отримати завдання, оброблені в одній або кількох попередніх сесіях, необхідно передати список ідентифікаторів сесій у параметрі `previous_session_id`. Це особливо корисно у випадках, коли задача оптимізації не завершилася через будь-яку помилку в процесі, і виконання потрібно довести до кінця. Для цього потрібно вказати ті самі аргументи, що й при початковому виконанні, разом зі списком `previous_session_id`, як описано вище.
> **Caution:** Завантаження даних із попередніх сесій (для відновлення оптимізації) може тривати до однієї години.
## `assets`
Дані мають бути структуровані як JSON-об'єкт, що містить інформацію про ціни закриття фінансових активів на певні дати. Формат такий:

- Первинний ключ (рядок): Назва або тикер фінансового активу (наприклад, "8801.T").
- Вторинний ключ (рядок): Дата у форматі YYYY-MM-DD.
- Значення (число): Ціна закриття активу на вказану дату. Ціни можна вводити як нормалізовані, так і ненормалізовані.

*Зверни увагу, що всі словники мають мати однаковий вторинний ключ (дати). Якщо певному активу бракує дати, наявної в інших, дані слід доповнити для забезпечення однорідності. Наприклад, це можна зробити, використавши останню відому ціну закриття цього активу.*
### Приклад
``` py
{
    "8801.T": {
        "2023-01-01": 2374.0,
        "2023-01-02": 2374.0,
        "2023-01-03": 2374.0,
        "2023-01-04": 2356.5,
        ...
    },
    "AAPL": {
        "2023-01-01": 145.2,
        "2023-01-02": 146.5,
        "2023-01-03": 147.3,
        "2023-01-04": 148.1,
        ...
    },
    ...
}
```

In [ ]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# Access function
dpo_solver = catalog.load("global-data-quantum/quantum-portfolio-optimizer")

> **Note:** Дані про активи мають містити щонайменше ціни закриття для ``(nt+1) * dt`` (дивись розділ вхідних даних [`qubo_settings`](#qubo_settings)) часових міток (наприклад, днів).
## `qubo_settings`
Наступна таблиця описує ключі словника `qubo_settings`. Побудуй словник, вказавши кількість часових кроків `nt`, кількість qubits роздільної здатності `nq` та `max_investment` — або зміни інші значення за замовчуванням.
| Назва               | Тип     | Опис                                                                         | Обов'язковий | За замовчуванням | Приклад |
|---------------------|---------|------------------------------------------------------------------------------|--------------|------------------|---------|
| nt                  | int     | Кількість часових кроків                                                     | Так          | -                | 4       |
| nq                  | int     | Кількість qubits роздільної здатності                                        | Так          | -                | 4       |
| max_investment      | float   | Максимальна кількість одиниць валюти, що інвестується в усі активи           | Так          | -                | 10      |
| dt*                  | int     | Часове вікно, що розглядається на кожному часовому кроці. Одиниця відповідає інтервалам між ключами в даних про активи | Ні           | 30               | -       |
| risk_aversion       | float     | Коефіцієнт несприйняття ризику                                             | Ні           | 1000             | -       |
| transaction_fee     | float     | Коефіцієнт транзакційного збору                                            | Ні           | 0.01             | -       |
| restriction_coeff   | float   | Множник Лагранжа, що використовується для виконання обмеження задачі у QUBO-формуляції | Ні           | 1                | -       |
## `ansatz_settings`
Щоб змінити параметри за замовчуванням, створи словник для параметра `ansatz_settings` з такими ключами. За замовчуванням ансатц встановлений на `"real_amplitudes"`, а обидва додаткові параметри (дивись таблицю нижче) мають значення `False`.
| Назва                 | Тип      | Опис                                                                         | Обов'язковий | За замовчуванням |
|-----------------------|----------|------------------------------------------------------------------------------|--------------|------------------|
| ansatz[*](#single-star)                | str      | Ансатц для використання                                        | Ні           | `"real_amplitudes"` |
| multiple_passmanager[**](#double-star)  | bool     | Вмикає підпрограму multiple passmanager (недоступна для Tailored ansatz)  | Ні           | `False`   |
| dd_enable   | bool     | Додає динамічне роз'єднання                                                  | Ні           | `False`   |

<span id="single-star"></span>
\* Доступні ансатци
- `real_amplitudes`
- `cyclic`
- `optimized_real_amplitudes`
- `tailored` (лише для бекенду `ibm_torino`, 7 активів, 4 часові кроки та 4 qubits роздільної здатності)

<span id="double-star"></span>
** Якщо ``multiple_passmanager`` встановлено в ``False``, функція використовує стандартний менеджер проходів Qiskit з `optimization_level=3`. Якщо встановлено в ``True``, підпрограма ``multiple_passmanager`` порівнює три менеджери проходів: попередній стандартний менеджер проходів Qiskit, менеджер проходів, що відображає qubits на ланцюжок найближчих сусідів QPU, та сервіси [AI transpiler](/guides/ai-transpiler-passes). Потім обирається менеджер проходів з оцінено нижчою накопиченою похибкою.
## `optimizer_settings`
Цей параметр — словник із деякими налаштовуваними параметрами процесу оптимізації.
| Назва              | Тип    | Опис                                           | Обов'язковий | За замовчуванням      |
|--------------------|--------|------------------------------------------------|--------------|----------------------|
| primitive_options  | json   | Налаштування примітиву                         | Ні           | -                    |
| optimizer          | str    | Вибраний класичний оптимізатор                 | Ні           | `"differential_evolution"` |
| optimizer_options  | json   | Конфігурація оптимізатора                      | Ні           | -                    |
> **Note:** Наразі єдиний доступний варіант оптимізатора — `"differential_evolution"`.

Під ключами `primitive_options` та `optimizer_options` задаються словники з такими параметрами:
### `primitive_options`
| Назва             | Тип    | Опис                                                | Обов'язковий | За замовчуванням          | Приклад                    |
|-------------------|--------|-----------------------------------------------------|--------------|---------------------------|----------------------------|
| sampler_shots     | int    | Кількість вимірів Sampler.                          | Ні           | 100000                    | -                          |
| estimator_shots   | int    | Кількість вимірів Estimator.                        | Ні           | 25000                     | -                          |
| estimator_precision | float | Бажана точність очікуваного значення. Якщо вказано, замість `estimator_shots` буде використана точність. | Ні | `None` | 0.015625 · (1 / sqrt(4096)) |
| max_time          | int або str | Максимальний час, протягом якого сесія runtime може залишатися відкритою до примусового закриття. Можна вказати в секундах (int) або у вигляді рядка, наприклад `"2h 30m 40s"`. Має бути менше системного максимуму. | Ні | `None` | `"1h 15m"`                |
### `optimizer_options`
| Назва             | Тип      | Опис                                         | Обов'язковий | За замовчуванням |
|-------------------|----------|----------------------------------------------|--------------|-----------------|
| num_generations   | int      | Кількість поколінь                           | Ні           | 20              |
| population_size   | int      | Розмір популяції                             | Ні           | 20              |
| mutation_range    | list     | Максимальний та мінімальний коефіцієнт мутації | Ні           | [0, 0.25]       |
| recombination     | float    | Коефіцієнт рекомбінації                      | Ні           | 0.4             |
| max_parallel_jobs | int      | Максимальна кількість паралельно виконуваних завдань QPU | Ні           | 3               |
| max_batchsize | int      | Максимальний розмір пакету                   | Ні           | 200             |
> **Note:** - Кількість поколінь, що оцінюються диференціальною еволюцією, дорівнює `num_generations` + 1, оскільки початкова популяція включена.
> - Загальна кількість схем обчислюється як `(num_generations + 1) * population_size`.
> - Використання більшого розміру популяції та більшої кількості поколінь загалом покращує якість результатів оптимізації. Однак не рекомендується перевищувати розмір популяції 120 та кількість поколінь більше 20 (наприклад, ``120 * 21 = 2520`` схем загалом), оскільки це генерує надмірну кількість схем, що може бути обчислювально витратним і трудомістким.
> 
> - Функція дозволяє відновити попередню оптимізацію, і завжди можна збільшити кількість поколінь (надаючи ті самі вхідні дані, крім `previous_session_id` та збільшеного ``num_generations``).
> **Note:** Забезпечуй дотримання обмежень на завдання Qiskit Runtime.
> 
> - Sampler: `sampler_shots <= 10_000_000`.
> - Estimator: `max_batchsize * estimator_shots * observable_size <= 10_000_000` (для цієї функції всі члени спостережуваної величини комутують, тому `observable_size=1`).
> 
> Дивись посібник [Обмеження завдань](/guides/job-limits) для отримання додаткової інформації.
# Вихідні дані
Функція повертає два словники: словник `"result"`, що містить найкращі результати оптимізації, включаючи оптимальне рішення та його мінімальне значення цільової функції; та `"metadata"` — з даними про всі результати, отримані під час процесу оптимізації, разом із відповідними метриками.

Перший словник зосереджений на найефективнішому рішенні, тоді як другий надає детальну інформацію про всі рішення, включаючи значення цільових функцій та інші відповідні метрики.

## Вихідні словники:
| **Назва**   | **Тип**                      | **Опис**                                                                        | **Приклад**  |
|-------------|------------------------------|---------------------------------------------------------------------------------|--------------|
| result      | dict[str, dict[str, float]]  | Містить інвестиційну стратегію в часі, де кожна часова мітка відповідає часткам інвестицій у конкретні активи (кожна частка — це сума інвестиції, нормована на загальну суму). | `{'time_1': {'asset_1': 0.2, 'asset_2': 0.3, ...}, ...}` |
| metadata    | dict[str, Any]               | Дані, згенеровані під час аналізу, включаючи рішення, витрати та метрики.       | Дивись приклади нижче |
### Опис словника `metadata`
| **Назва**                | **Тип**               | **Опис**                                                                                        | **Приклад**                  |
|--------------------------|-----------------------|-------------------------------------------------------------------------------------------------|------------------------------|
| session_id               | str                   | Унікальний ідентифікатор сесії IBM Quantum.                             | `"d0h30qjvpqf00084fgw0"`     |
| all_samples_metrics | dict                 | Словник, що містить різні метрики для кожного постобробленого зразка, наприклад витрати або обмеження. | Дивись опис нижче            |
| sampler_counts           | dict[str, int]        | Словник, де ключами є бітрядкові представлення відібраних рішень, а значеннями — їх кількості. | `{"101010": 3, "111000": 1}` |
| asset_order              | list[str]             | Список із відповідним порядком інвестування активів на кожному часовому кроці в інвестиційних стратегіях. | `["Asset_0", "Asset_1", "Asset_3"]`     |
| QUBO                     | list[list[float]]     | QUBO-матриця задачі.                                                                            | `[[-6.96e-01, 5.81e-01, -1.26e-02, 0.00e+00], ...]`     |
| resource_summary           | dict[str, dict[str, float]] | Зведення часу використання CPU та QPU (у секундах) на різних етапах процесу.        | `{'RUNNING: EXECUTING_QPU': {'CPU_TIME': 412.84, 'QPU_TIME': 87.22}, ...}` |
### Опис словника `all_samples_metrics`
| **Назва**               | **Тип**        | **Опис**                                                                                                 | **Приклад**                  |
|-------------------------|----------------|----------------------------------------------------------------------------------------------------------|------------------------------|
| investment_trajectories | list[list]     | Інвестиційні стратегії, отримані з декодованих квантових станів. | `[[1, 2, 2], [1, 2, 1]]`     |

| counts                  | list[int]      | Кількість разів, коли кожна інвестиційна траєкторія була відібрана. Індекс відповідає `investment_trajectories`. | `[5, 3]`                     |
| objective_costs         | list[float]    | Значення цільової функції для кожної інвестиційної траєкторії, впорядковані від найменшого до найбільшого. | `[0.98, 1.25]`               |
| sharpe_ratios           | list[float]    | Скоригована на ризик ефективність (коефіцієнт Шарпа) для кожної інвестиційної траєкторії. Вирівняно за індексом. | `[1.1, 0.7]`                 |
| returns                 | list[float]    | Очікувана прибутковість для кожної інвестиційної траєкторії. Вирівняно за індексом.                      | `[0.15, 0.10]`               |
| rest_breaches           | list[float]    | Максимальне відхилення від обмеження в межах кожної інвестиційної траєкторії. Вирівняно за індексом.    | `[0.0, 0.25]`                |
| transaction_costs       | list[float]    | Розрахована транзакційна вартість, пов'язана з кожною інвестиційною траєкторією. Вирівняно за індексом. | `[0.01, 0.02]`               |
# Початок роботи
Пройди автентифікацію за допомогою свого [API-ключа](https://quantum.cloud.ibm.com/) та вибери Qiskit Function таким чином. (Цей фрагмент передбачає, що ти вже [зберіг свій обліковий запис](/guides/functions#install-qiskit-functions-catalog-client) у локальному середовищі.)

In [ ]:
import os
import glob
import pandas as pd


def read_and_join_csv(file_pattern):
    """
    Reads multiple CSV files matching the file pattern and combines them into a single DataFrame.

    Parameters:
    file_pattern (str): The pattern to match CSV files.

    Returns:
    pd.DataFrame: Combined DataFrame with data from all CSV files.
    """
    # Find all files matching the pattern
    csv_files = glob.glob(file_pattern)
    # Get the base file names without the .csv extension
    file_names = [os.path.basename(f).replace(".csv", "") for f in csv_files]
    # Read each CSV file into a DataFrame and set the first column as the index
    df_list = [pd.read_csv(f).set_index("Unnamed: 0") for f in csv_files]

    # Rename columns in each DataFrame to the base file names
    for df, name in zip(df_list, file_names):
        df.columns = [name]

    # Combine all DataFrames into one by merging them side by side
    combined_df = pd.concat(df_list, axis=1)
    return combined_df


file_pattern = "route/to/folder/with/assets/data/*.csv"
assets = read_and_join_csv(file_pattern).to_dict()

## Приклад: динамічна оптимізація портфеля з сімома активами
Цей приклад демонструє, як виконати функцію динамічної оптимізації портфеля (DPO) та налаштувати її параметри для оптимальної продуктивності. Він включає детальні кроки з тонкого налаштування параметрів для досягнення бажаних результатів.

Цей випадок включає сім активів, чотири часові кроки та чотири qubits роздільної здатності, що разом потребує 112 qubits.
### 1. Зчитай активи, що входять до портфеля.
Якщо всі активи портфеля зберігаються в папці за певним шляхом, ти можеш завантажити їх у `pandas.DataFrame` та перетворити на об'єкт у форматі `dict` за допомогою такої функції.

In [ ]:
qubo_settings = {
    "nt": 4,
    "nq": 4,
    "dt": 30,
    "max_investment": 25,
    "risk_aversion": 1000.0,
    "transaction_fee": 0.01,
    "restriction_coeff": 1.0,
}

Для цього прикладу ми використали активи [8801.T](https://finance.yahoo.com/quote/8801.T), [CLF](https://finance.yahoo.com/quote/CLF), [GBPJPY](https://finance.yahoo.com/quote/GBPJPY), [ITX.MC](https://finance.yahoo.com/quote/ITX.MC), [META](https://finance.yahoo.com/quote/META), [TMBMKDE-10Y](https://finance.yahoo.com/quote/TMBMKDE-10Y) та [XS2239553048](https://finance.yahoo.com/quote/XS2239553048). Наступний графік ілюструє дані, використані в цьому прикладі, показуючи динаміку денних цін закриття активів з 1 січня по 1 вересня 2023 року.

У цьому прикладі, щоб забезпечити однорідність дат, ми заповнили неторгові дні ціною закриття з попередньої доступної дати. Ми застосовуємо цей крок, оскільки обрані активи належать до різних ринків із різними торговими днями, тому стандартизація набору даних є необхідною для узгодженості.
![Візуалізація історичних даних активів](../docs/images/guides/global-data-quantum-optimizer/assets.avif)

### 2. Визнач задачу.

Визнач специфікації задачі, налаштувавши параметри в словнику `qubo_settings`.

In [ ]:
optimizer_settings = {
    "de_optimizer_settings": {
        "num_generations": 20,
        "population_size": 90,
        "recombination": 0.4,
        "max_parallel_jobs": 5,
        "max_batchsize": 4,
        "mutation_range": [0.0, 0.25],
    },
    "optimizer": "differential_evolution",
    "primitive_settings": {
        "estimator_shots": 25_000,
        "estimator_precision": None,
        "sampler_shots": 100_000,
    },
}

### 3. Визнач налаштування оптимізатора та ансатцу. (Необов'язково)
За бажанням визнач конкретні вимоги до процесу оптимізації, включаючи вибір оптимізатора та його параметрів, а також специфікацію примітиву та його конфігурацій.

Для Tailored Ansatz обраний розмір популяції базується на попередніх експериментах, які показали, що це значення забезпечує стабільну та ефективну оптимізацію.

У випадку Real Amplitudes Ansatz можна дотримуватися лінійного зв'язку між ``population_size`` та кількістю qubits у схемі. Як наближене правило, рекомендується використовувати **мінімум** ``population_size ~ 0.8 * n_qubits`` для ансатцу `real_amplitudes`.

Очікується, що Optimized Real Amplitudes матиме кращу продуктивність оптимізації, ніж ансатц Real Amplitudes. Однак кількість змінних для оптимізації в цьому ансатці зростає значно швидше, ніж у випадку Real Amplitudes (дивись [статтю](https://arxiv.org/pdf/2412.19150)). Тому для великих задач Optimized Real Amplitudes потребує більше виконань схем. Optimized Real Amplitudes, мабуть, буде корисним для задач до 100 qubits, але рекомендується бути уважним при встановленні параметрів ``population_size``. Як приклад такого масштабування ``population_size``: для задачі з 84 qubits Optimize Real Amplitudes потребує 120 ``population_size``, тоді як для задачі з 56 qubits достатньо ``population_size`` 40.

In [ ]:
ansatz_settings = {
    "ansatz": "tailored",
    "multiple_passmanager": False,
}

Також можна вибрати конкретний ансатц. Нижче використовується ансатц ``'Tailored'``.

In [ ]:
dpo_job = dpo_solver.run(
    assets=assets,
    qubo_settings=qubo_settings,
    optimizer_settings=optimizer_settings,
    ansatz_settings=ansatz_settings,
    backend_name="<backend name>",
    previous_session_id=[],
    apply_postprocess=True,
)

### 4. Запусти задачу.

In [ ]:
# Get the results of the job
dpo_result = dpo_job.result()

# Show the solution strategy
dpo_result["result"]

{'time_step_0': {'8801.T': 0.11764705882352941,
  'ITX.MC': 0.20588235294117646,
  'META': 0.38235294117647056,
  'GBPJPY=X': 0.058823529411764705,
  'TMBMKDE-10Y': 0.0,
  'CLF': 0.058823529411764705,
  'XS2239553048': 0.17647058823529413},
 'time_step_1': {'8801.T': 0.11428571428571428,
  'ITX.MC': 0.14285714285714285,
  'META': 0.2,
  'GBPJPY=X': 0.02857142857142857,
  'TMBMKDE-10Y': 0.42857142857142855,
  'CLF': 0.0,
  'XS2239553048': 0.08571428571428572},
 'time_step_2': {'8801.T': 0.0,
  'ITX.MC': 0.09375,
  'META': 0.3125,
  'GBPJPY=X': 0.34375,
  'TMBMKDE-10Y': 0.0,
  'CLF': 0.0,
  'XS2239553048': 0.25},
 'time_step_3': {'8801.T': 0.3939393939393939,
  'ITX.MC': 0.09090909090909091,
  'META': 0.12121212121212122,
  'GBPJPY=X': 0.18181818181818182,
  'TMBMKDE-10Y': 0.0,
  'CLF': 0.0,
  'XS2239553048': 0.21212121212121213}}

### 5. Отримай результати.
Як зазначено в розділі [Вихідні дані](#output), функція повертає словник з інвестиційними траєкторіями, впорядкованими від найменшого до найбільшого за значенням цільової функції. Цей набір результатів дозволяє визначити траєкторію з найнижчою вартістю та відповідні інвестиційні оцінки. Крім того, він забезпечує аналіз різних траєкторій, полегшуючи вибір тих, що найкраще відповідають конкретним потребам або цілям. Така гнучкість дозволяє адаптувати вибір до різноманітних уподобань або сценаріїв.
Почни з відображення результуючої стратегії, яка досягла найнижчої цільової вартості, знайденої в процесі.

In [ ]:
# Convert metadata to a DataFrame
df = pd.DataFrame(dpo_result["metadata"]["all_samples_metrics"])

# Find the minimum objective cost
min_cost = df["objective_costs"].min()
print(f"Minimum Objective Cost Found: {min_cost:.2f}")

# Extract the row with the lowest cost
best_row = df[df["objective_costs"] == min_cost].iloc[0]

# Display the results associated with the best solution
print("Best Solution:")
print(f"  - Restriction Deviation: {best_row['rest_breaches']}%")
print(f"  - Sharpe Ratio: {best_row['sharpe_ratios']:.2f}")
print(f"  - Return: {best_row['returns']}")

Minimum Objective Cost Found: -3.78
Best Solution:
  - Restriction Deviation: 40.0
  - Sharpe Ratio: 24.82
  - Return: 0.46


### 6. Performance analysis

Last, analyze the performance of your optimization application. Specifically, compare your results, obtained in the previous example, against a random baseline to assess the effectiveness of our approach. If the quantum algorithm demonstrably and consistently produces results with lower cost values, it indicates an effective optimization process.

The figure presents the probability distributions of the objective costs. To generate these distributions, take the list of objective costs from the function result and count the occurrences of each cost value (values rounded to the second decimal place). Then, update the count column accordingly by joining counts of identical rounded values. Note that, for better visual comparison, the occurrence counts have been normalized so that each distribution is displayed between 0 and 1.

![Visualization of the solution of the optimization](../docs/images/guides/global-data-quantum-optimizer/cost_distribution.svg)

As shown in the figure (blue solid line), the cost distribution for our Variational Quantum Eigensolver (post-processed with SQD) approach is sharply concentrated at lower objective cost values, indicating good optimization performance. In contrast, the noisy baseline exhibits a broader distribution, centered around higher cost values. The gray dashed vertical line represents the mean value of the random distribution, further highlighting the consistency of the function in returning optimized investment strategies. For an additional comparison, the black dashed line in the figure corresponds to the solution obtained with the Gurobi optimizer (free version). All these results are further explored in the benchmarks below for the "Mixed Assets" example evaluated with the "Tailored" ansatz.

# Benchmarks

This function was tested under different configurations of resolution qubits, ansatz circuits, and groupings of assets from various sectors: a mix of different assets (Set 1), oil derivatives (Set 2), and IBEX35 (Set 3). See more details in the following table.

| Set       | Date       | Assets                                                                 |
|-----------|------------|------------------------------------------------------------------------|
| **Set 1** | 01/01/2023 | 8801.T, CL=F, GBPJPY=X, ITX.MC, META, TMBMKDE-10Y, XS2239553048         |
| **Set 2** | 01/06/2023 | CL=F, BZ=F, HO=F, NG=F, XOM, RB=F, 2222.SR                               |
| **Set 3** | 01/11/2022 | ACS.MC, ITX.MC, FER.MC, ELE.MC, SCYR.MC, AENA.MC, AMS.MC                |

Two key metrics were used to evaluate solution quality.
1. The objective cost, which measures optimization efficiency by comparing the cost function value from each experiment with results from Gurobi (free version).
2. The Sharpe ratio, which captures the risk-adjusted return of each portfolio, offering insight into the financial performance of the solutions.

Together, these metrics benchmark both computational and financial aspects of the quantum-generated portfolios.


| Example             | Qubits | Ansatz                  | Depth | Runtime Usage (s) | Total usage (s) | Objective cost | Sharpe | Gurobi objective cost | Gurobi Sharpe |
|-------------------------------|--------|-------------------------------|--------|-------------------|------------------|----------------|--------|------------------------|----------------|
| Mixed Assets (Set 1, 4 time steps, 4-bit)   | 112    | Tailored                      | 83     | 12735             | 13095            | -3.78          | 24.82  | -4.25                  | 24.71          |
| Mixed Assets (Set 1,4 time steps, 4 time steps, 4-bit)   | 112    | Real Amplitudes               | 359    | 11739             | 11903            | -3.39          | 23.64  | -4.25                  | 24.71          |
| Oil Derivatives (Set 2, 4 time steps, 3-bit)| 84     | Optimized Real Amplitudes     | 78     | 6180              | 6350             | -3.73          | 19.13  | -4.19                  | 21.71          |
| IBEX35 (Set 3, 4 time steps, 2-bit)         | 56     | Optimized Real Amplitudes     | 96     | 3314              | 3523             | -3.67          | 14.48  | -4.11                  | 16.44          |



Results show that the quantum optimizer, with problem-specific ansatzes, effectively identifies efficient investment strategies across various portfolio types.

Below we detail both the population size and the number of generations specified in the `optimizer_options` dictionary. All other parameters were set to their default values.

| **Example**                | ``population_size`` | ``num_generations``   |
|----------------------------|----------------------|----------------------|
| Mixed Assets Portfolio     | 90                   | 20                   |
| Mixed Assets Portfolio     | 92                   | 20                   |
| Oil Derivatives Portfolio  | 120                  | 20                   |
| IBEX35 Portfolio           | 40                   | 20                   |

The number of generations was set to 20, as this value was found to be sufficient to reach convergence. Additionally, the default values for the optimizer's internal parameters were left unchanged, as they consistently provided good performance and are generally recommended by the literature and implementation guidelines.

# Get support
If you need help, you can send an email to qpo.support@globaldataquantum.com. In your message, provide the function job ID.

## Next steps

<Admonition type="note" title="Recommendations">
*   Read [the associated research paper](https://arxiv.org/pdf/2412.19150).
*   Request access to the function by filling in [this form](https://www.globaldataquantum.com/en/quantum-portfolio-optimizer/#form).
*   Try the [Dynamic Portfolio Optimization](/docs/tutorials/global-data-quantum-optimizer) tutorial.

</Admonition>